# 🗣️ Indic Parler-TTS — Full Voice Control Studio

**Generate natural-sounding Indian language TTS with full control over voice characteristics.**

| Feature | Details |
|---------|----------|
| **Model** | [ai4bharat/indic-parler-tts](https://huggingface.co/ai4bharat/indic-parler-tts) (938M params) |
| **Languages** | 21 Indian languages + English |
| **Speakers** | 69 unique voices |
| **Controls** | Pitch, speed, expressivity, emotion, noise, reverb, quality |
| **Runtime** | Optimized for **T4 GPU** |

### 🔄 Workflow
1. Install dependencies → Authenticate → Load model
2. Configure voice (speaker, pitch, speed, emotion, etc.)
3. Upload text → Generate audio → Download

---

## ⚙️ Cell 1 — Install Dependencies

In [1]:
# ============================================================
# CELL 1 — Install required packages
# ============================================================

!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q transformers accelerate soundfile pydub ipywidgets
!pip install -q "protobuf>=3.20,<5"
!apt-get -qq install -y ffmpeg > /dev/null 2>&1

print("✅ All dependencies installed!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.9 MB/s 

---
## 🔑 Cell 2 — HuggingFace Authentication

This model is **gated**. You need to:
1. Go to [ai4bharat/indic-parler-tts](https://huggingface.co/ai4bharat/indic-parler-tts) and accept the terms
2. Create a token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Enter it below

In [2]:
# ============================================================
# CELL 2 — Authenticate with HuggingFace
# ============================================================

from huggingface_hub import notebook_login
notebook_login()

---
## 🧠 Cell 3 — Load Model & Tokenizers

Loads the model with **SDPA attention** (fastest on T4 without flash-attn).  
Uses `float16` for optimal T4 VRAM usage (~3.75 GB model).

In [3]:
# ============================================================
# CELL 3 — Load model and tokenizers
# ============================================================

import torch
import numpy as np
import soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

MODEL_NAME = "ai4bharat/indic-parler-tts"

# --- Device setup ---
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device.startswith("cuda") else torch.float32

print(f"🖥️  Device: {device}")
if device.startswith("cuda"):
    gpu_name = torch.cuda.get_device_name(0)
    print(f"   GPU: {gpu_name}")

# --- Load model with SDPA attention ---
print(f"\n⏳ Loading model: {MODEL_NAME}")
print("   Using eager attention as SDPA is not supported for T5EncoderModel...")

model = ParlerTTSForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager", # Changed from "sdpa" to "eager"
    torch_dtype=torch_dtype,
).to(device)

# --- Load tokenizers ---
# indic-parler-tts uses TWO tokenizers:
#   description_tokenizer = for the voice description prompt
#   prompt_tokenizer      = for the text to speak
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
prompt_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

SAMPLE_RATE = model.config.sampling_rate

print(f"\n✅ Model loaded successfully!")
print(f"   Sample rate: {SAMPLE_RATE} Hz")
print(f"   Dtype: {torch_dtype}")
print(f"   Attention: eager") # Updated print statement


🖥️  Device: cuda:0
   GPU: Tesla T4

⏳ Loading model: ai4bharat/indic-parler-tts
   Using eager attention as SDPA is not supported for T5EncoderModel...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.75G [00:00<?, ?B/s]

  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

  "_name_or_path": "ylacombe/dac_44khz",
  "architectures": [
    "DacModel"
  ],
  "codebook_dim": 8,
  "codebook_loss_weight": 1.0,
  "codebook_size": 1024,
  "commitment_loss_weight": 0.25,
  "decoder_hidden_si

...

tokenizer_config.json:   0%|          | 0.00/990 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/10.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]


✅ Model loaded successfully!
   Sample rate: 44100 Hz
   Dtype: torch.float16
   Attention: eager


---
## 🎛️ Cell 4 — Voice Configuration Panel

Configure **every aspect** of your TTS voice using the interactive controls below.  
The voice description is **auto-generated** from your selections, or you can override it with a custom description.

### How it works
- Select a **language** → speaker list filters automatically
- Adjust **pitch, speed, expressivity, emotion** etc.
- The description text updates live — or write your own!
- Tweak **generation parameters** (temperature, sampling) for variation

In [4]:
# ============================================================
# CELL 4 — Voice Configuration Panel (Interactive Widgets)
# ============================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# =====================================================
# SPEAKER DATABASE — All 69 speakers by language
# First speaker in each list is the recommended default
# =====================================================
SPEAKERS_BY_LANGUAGE = {
    "Hindi":    ["Rohit", "Divya", "Aman", "Rani"],
    "English":  ["Thoma", "Mary", "Kabir", "Priya", "Raghav"],
    "Bengali":  ["Arjun", "Aditi", "Tapan", "Rashmi", "Arnav", "Riya"],
    "Marathi":  ["Sanjay", "Sunita", "Nikhil", "Radha", "Varun", "Isha"],
    "Tamil":    ["Kavitha", "Jaya"],
    "Telugu":   ["Prakash", "Lalitha", "Kiran"],
    "Gujarati": ["Hardik", "Meera"],
    "Kannada":  ["Suresh", "Lakshmi", "Naveen"],
    "Malayalam":["Arun", "Anjali", "Deepa"],
    "Odia":     ["Bikram", "Priti"],
    "Assamese": ["Bhaskar", "Priyanka"],
    "Punjabi":  ["Gurpreet", "Harleen"],
    "Urdu":     ["Imran", "Amrita"],
    "Nepali":   ["Sagar", "Nisha"],
    "Sanskrit": ["Rishi", "Sita"],
    "Konkani":  ["Karan", "Leela"],
    "Maithili": ["Maya", "Mohan"],
    "Dogri":    ["Krishan", "Pooja"],
    "Bodo":     ["Gyan", "Rekha"],
    "Manipuri": ["Tomba", "Sanahanbi"],
    "Santali":  ["Soren", "Champa"],
    "Sindhi":   ["Govind", "Sindhu"],
}

ALL_LANGUAGES = sorted(SPEAKERS_BY_LANGUAGE.keys())

# Languages with official emotion support
EMOTION_LANGUAGES = {
    "Assamese", "Bengali", "Bodo", "Dogri", "Kannada",
    "Malayalam", "Marathi", "Sanskrit", "Nepali", "Tamil"
}

EMOTIONS = [
    "Neutral", "Narration", "Conversation", "Happy", "Sad",
    "News", "Command", "Anger", "Disgust", "Fear", "Surprise"
]

# =====================================================
# CREATE WIDGETS
# =====================================================

style = {'description_width': '140px'}
layout = widgets.Layout(width='500px')
wide_layout = widgets.Layout(width='600px')
slider_layout = widgets.Layout(width='500px')

# --- Voice Identity ---
w_language = widgets.Dropdown(
    options=ALL_LANGUAGES,
    value="Hindi",
    description="🌐 Language:",
    style=style, layout=layout
)

w_speaker = widgets.Dropdown(
    options=SPEAKERS_BY_LANGUAGE["Hindi"],
    value="Rohit",
    description="🎤 Speaker:",
    style=style, layout=layout
)

# --- Voice Characteristics ---
w_pitch = widgets.Dropdown(
    options=["low", "slightly low", "moderate", "slightly high", "high"],
    value="moderate",
    description="🎵 Pitch:",
    style=style, layout=layout
)

w_speed = widgets.Dropdown(
    options=["slow", "slightly slow", "moderate", "slightly fast", "fast"],
    value="moderate",
    description="⏩ Speaking Rate:",
    style=style, layout=layout
)

w_expressivity = widgets.Dropdown(
    options=["monotone", "slightly monotone", "slightly expressive", "expressive", "very expressive"],
    value="slightly expressive",
    description="🎭 Expressivity:",
    style=style, layout=layout
)

w_emotion = widgets.Dropdown(
    options=EMOTIONS,
    value="Neutral",
    description="😊 Emotion:",
    style=style, layout=layout
)

# --- Recording Characteristics ---
w_noise = widgets.Dropdown(
    options=["no background noise", "minimal ambient noise", "slightly noisy", "noisy"],
    value="no background noise",
    description="🔇 Background Noise:",
    style=style, layout=layout
)

w_reverb = widgets.Dropdown(
    options=["very close", "close", "slightly distant", "distant"],
    value="very close",
    description="📏 Reverberation:",
    style=style, layout=layout
)

w_quality = widgets.Dropdown(
    options=["clear audio", "very clear audio", "high quality recording", "excellent quality recording"],
    value="very clear audio",
    description="💎 Audio Quality:",
    style=style, layout=layout
)

# --- Chunking & Output ---
w_chunk_size = widgets.IntSlider(
    value=250, min=100, max=500, step=25,
    description="📏 Chunk Size (chars):",
    style=style, layout=slider_layout
)

w_silence = widgets.FloatSlider(
    value=0.4, min=0.1, max=2.0, step=0.1,
    description="⏸️ Silence Gap (sec):",
    style=style, layout=slider_layout
)

w_output_format = widgets.ToggleButtons(
    options=["mp3", "wav"],
    value="mp3",
    description="💾 Format:",
    style=style
)

# --- Generation Parameters (Advanced) ---
w_do_sample = widgets.Checkbox(
    value=True,
    description="🎲 Sampling (do_sample)",
    style=style, layout=layout
)

w_temperature = widgets.FloatSlider(
    value=1.0, min=0.1, max=2.0, step=0.05,
    description="🌡️ Temperature:",
    style=style, layout=slider_layout
)

w_use_compile = widgets.Checkbox(
    value=False,
    description="⚡ torch.compile (slow warmup, faster generation)",
    style={'description_width': '350px'},
    layout=widgets.Layout(width='600px')
)

# --- Custom Description Override ---
w_custom_desc = widgets.Textarea(
    value="",
    placeholder="Leave empty to auto-generate, or type your own voice description here...",
    description="✏️ Custom Override:",
    style=style,
    layout=widgets.Layout(width='600px', height='80px')
)

# --- Live Preview ---
w_preview = widgets.HTML(value="")

# =====================================================
# AUTO-UPDATE LOGIC
# =====================================================

def update_speakers(*args):
    """Filter speakers when language changes."""
    lang = w_language.value
    speakers = SPEAKERS_BY_LANGUAGE.get(lang, ["Speaker"])
    w_speaker.options = speakers
    w_speaker.value = speakers[0]
    # Show emotion warning for unsupported languages
    if lang not in EMOTION_LANGUAGES:
        w_emotion.description = "😊 Emotion (⚠️):"
    else:
        w_emotion.description = "😊 Emotion:"
    update_preview()

def build_description():
    """Auto-compose voice description from widget values."""
    speaker = w_speaker.value
    pitch = w_pitch.value
    speed = w_speed.value
    expressivity = w_expressivity.value
    emotion = w_emotion.value
    noise = w_noise.value
    reverb = w_reverb.value
    quality = w_quality.value

    # Build the emotion/style phrase
    if emotion == "Neutral":
        emotion_phrase = ""
    else:
        emotion_phrase = f" in a {emotion.lower()} tone"

    desc = (
        f"{speaker}'s voice delivers a {expressivity} speech{emotion_phrase} "
        f"with a {speed} pace and {pitch} pitch. "
        f"The recording is of {quality}, "
        f"with the speaker's voice sounding {reverb} "
        f"with {noise}."
    )
    return desc

def get_active_description():
    """Return custom description if set, otherwise auto-generated."""
    if w_custom_desc.value.strip():
        return w_custom_desc.value.strip()
    return build_description()

def update_preview(*args):
    """Update the live preview HTML."""
    desc = get_active_description()
    is_custom = bool(w_custom_desc.value.strip())
    source = "✏️ <b>Custom</b>" if is_custom else "🤖 <b>Auto-generated</b>"
    emotion_warn = ""
    if w_language.value not in EMOTION_LANGUAGES and w_emotion.value != "Neutral":
        emotion_warn = "<br>⚠️ <i>Emotion support for this language is unofficial — results may vary.</i>"
    w_preview.value = (
        f'<div style="background:#1a1a2e; color:#e0e0e0; padding:12px; '
        f'border-radius:8px; margin-top:8px; border-left:4px solid #00d2ff;">'
        f'{source}<br><br>'
        f'<code style="color:#00d2ff; word-wrap:break-word; white-space:pre-wrap;">{desc}</code>'
        f'{emotion_warn}</div>'
    )

# --- Connect observers ---
w_language.observe(update_speakers, names='value')
for w in [w_speaker, w_pitch, w_speed, w_expressivity, w_emotion,
          w_noise, w_reverb, w_quality, w_custom_desc]:
    w.observe(update_preview, names='value')

# =====================================================
# DISPLAY THE PANEL
# =====================================================

header_style = 'style="color:#00d2ff; font-size:14px; margin:12px 0 4px 0; font-weight:bold;"'

display(HTML(f'<h2 style="color:#00d2ff;">🎛️ Voice Configuration Panel</h2>'))

display(HTML(f'<p {header_style}>🎤 Voice Identity</p>'))
display(w_language, w_speaker)

display(HTML(f'<p {header_style}>🎵 Voice Characteristics</p>'))
display(w_pitch, w_speed, w_expressivity, w_emotion)

display(HTML(f'<p {header_style}>🎙️ Recording Environment</p>'))
display(w_noise, w_reverb, w_quality)

display(HTML(f'<p {header_style}>📝 Description Override</p>'))
display(w_custom_desc)

display(HTML(f'<p {header_style}>📏 Chunking & Output</p>'))
display(w_chunk_size, w_silence, w_output_format)

display(HTML(f'<p {header_style}>⚙️ Generation Parameters (Advanced)</p>'))
display(w_do_sample, w_temperature, w_use_compile)

display(HTML(f'<p {header_style}>📋 Live Description Preview</p>'))
display(w_preview)

# Initial preview
update_preview()

print("\n✅ Configuration panel ready! Adjust settings above, then proceed.")

Dropdown(description='🌐 Language:', index=6, layout=Layout(width='500px'), options=('Assamese', 'Bengali', 'Bo…

Dropdown(description='🎤 Speaker:', layout=Layout(width='500px'), options=('Rohit', 'Divya', 'Aman', 'Rani'), s…

Dropdown(description='🎵 Pitch:', index=2, layout=Layout(width='500px'), options=('low', 'slightly low', 'moder…

Dropdown(description='⏩ Speaking Rate:', index=2, layout=Layout(width='500px'), options=('slow', 'slightly slo…

Dropdown(description='🎭 Expressivity:', index=2, layout=Layout(width='500px'), options=('monotone', 'slightly …

...

Checkbox(value=True, description='🎲 Sampling (do_sample)', layout=Layout(width='500px'), style=DescriptionStyl…

FloatSlider(value=1.0, description='🌡️ Temperature:', layout=Layout(width='500px'), max=2.0, min=0.1, step=0.0…

Checkbox(value=False, description='⚡ torch.compile (slow warmup, faster generation)', layout=Layout(width='600…

HTML(value='')


✅ Configuration panel ready! Adjust settings above, then proceed.


---
## 📂 Cell 5 — Upload Your Text File

In [5]:
# ============================================================
# CELL 5 — Upload text file + smart chunking
# ============================================================

from google.colab import files
import textwrap
import re

print("📂 Please upload your text file (.txt)")
print("   Supports: Devanagari (हिंदी) and/or Latin script (Hinglish)")
print("   Tip: UTF-8 encoded files work best.\n")

uploaded = files.upload()

if not uploaded:
    raise ValueError("❌ No file uploaded. Please re-run this cell and upload a .txt file.")

# --- Read the uploaded file ---
filename = list(uploaded.keys())[0]
raw_text = uploaded[filename].decode("utf-8", errors="replace")

# --- Text Preprocessing ---
cleaned_text = re.sub(r'\r\n|\r', '\n', raw_text)
cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)
cleaned_text = re.sub(r'[ \t]+', ' ', cleaned_text).strip()

# --- Smart Chunker (uses CHUNK_SIZE from widget) ---
CHUNK_SIZE = w_chunk_size.value

def smart_chunk(text, max_chars=CHUNK_SIZE):
    """Split text into chunks at sentence boundaries."""
    sentence_endings = re.compile(r'(?<=[।!?.])\s+')
    sentences = sentence_endings.split(text)

    chunks = []
    current = ""
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(current) + len(sentence) + 1 <= max_chars:
            current = (current + " " + sentence).strip()
        else:
            if current:
                chunks.append(current)
            if len(sentence) > max_chars:
                wrapped = textwrap.wrap(sentence, width=max_chars)
                chunks.extend(wrapped[:-1])
                current = wrapped[-1]
            else:
                current = sentence
    if current:
        chunks.append(current)
    return chunks

chunks = smart_chunk(cleaned_text, max_chars=CHUNK_SIZE)

print(f"\n✅ File loaded: {filename}")
print(f"   Total characters : {len(cleaned_text):,}")
print(f"   Chunk size       : {CHUNK_SIZE} chars")
print(f"   Total chunks     : {len(chunks)}")
print(f"\n📋 First 3 chunks preview:")
for i, chunk in enumerate(chunks[:3]):
    print(f"   [{i+1}] ({len(chunk)} chars): {chunk[:80]}...")

📂 Please upload your text file (.txt)
   Supports: Devanagari (हिंदी) and/or Latin script (Hinglish)
   Tip: UTF-8 encoded files work best.



Saving tts_test_comprehensive.txt to tts_test_comprehensive.txt

✅ File loaded: tts_test_comprehensive.txt
   Total characters : 17,615
   Chunk size       : 250 chars
   Total chunks     : 87

📋 First 3 chunks preview:
   [1] (249 chars): ================================================================ HINDI TTS TEST ...
   [2] (77 chars): 8 (Quick Test). Copy individual lines or whole sections as your PREVIEW_TEXT....
   [3] (248 chars): ================================================================  ==============...


---
## 🔊 Cell 6 — Generate TTS (Batch Processing)

Processes all chunks with your configured voice settings.  
**Progress is shown in real time.** Failed chunks are skipped with a warning.

In [ ]:
# ============================================================
# CELL 6 — TTS Generation (Batch) with full parameter control
# ============================================================

from tqdm.notebook import tqdm
import time

# --- Fix StaticCache compatibility with transformers >= 4.46 ---
# The parler-tts library references StaticCache.max_batch_size which was
# renamed to batch_size and then removed in newer transformers versions.
from transformers import StaticCache
if not hasattr(StaticCache, 'max_batch_size'):
    StaticCache.max_batch_size = property(lambda self: self.batch_size)

# --- Read configuration from widgets ---
VOICE_DESCRIPTION = get_active_description()
SPEAKER_NAME = w_speaker.value
DO_SAMPLE = w_do_sample.value
TEMPERATURE = w_temperature.value
USE_COMPILE = w_use_compile.value
SILENCE_DURATION = w_silence.value
OUTPUT_FORMAT = w_output_format.value

print("🎤 Active Voice Configuration:")
print(f"   Speaker      : {SPEAKER_NAME}")
print(f"   Language     : {w_language.value}")
print(f"   Pitch        : {w_pitch.value}")
print(f"   Speed        : {w_speed.value}")
print(f"   Expressivity : {w_expressivity.value}")
print(f"   Emotion      : {w_emotion.value}")
print(f"   Quality      : {w_quality.value}")
print(f"   do_sample    : {DO_SAMPLE}")
print(f"   Temperature  : {TEMPERATURE}")
print(f"   torch.compile: {USE_COMPILE}")
print(f"\n📝 Description:\n   {VOICE_DESCRIPTION}")


# ─── Helper: tokenise + guarantee attention_mask is always present ─────────
# Root cause of the warning:
#   T5 (flan-t5-large) as `description_tokenizer` may omit `attention_mask`
#   from its output when no explicit padding is requested, because T5's
#   `model_input_names` can exclude it.  Parler-TTS then sees
#   `prompt_attention_mask` but a None `attention_mask` and fires the warning.
# Fix: always pass `return_attention_mask=True` and fall back to an all-ones
#      tensor if the tokeniser still omits it.
def _tokenize_with_mask(tokenizer, text):
    """Tokenise *text* and guarantee that attention_mask is a real tensor."""
    enc = tokenizer(
        text,
        return_tensors="pt",
        return_attention_mask=True,  # explicit — never rely on the default
    ).to(device)
    # Safety-net: some T5 tokeniser versions still skip it
    if enc.get("attention_mask") is None:
        enc["attention_mask"] = torch.ones(
            enc["input_ids"].shape, dtype=torch.long, device=device
        )
    return enc


# --- Optional torch.compile ---
if USE_COMPILE and device.startswith("cuda"):
    print("\n⚡ Compiling model (this takes ~2 min on T4, but speeds up generation)...")
    model.generation_config.cache_implementation = "static"
    model.forward = torch.compile(model.forward, mode="default")
    # Warmup pass
    warmup_desc   = _tokenize_with_mask(description_tokenizer, "warmup")
    warmup_prompt = _tokenize_with_mask(prompt_tokenizer,      "warmup")
    with torch.no_grad():
        _ = model.generate(
            input_ids=warmup_desc["input_ids"],
            attention_mask=warmup_desc["attention_mask"],
            prompt_input_ids=warmup_prompt["input_ids"],
            prompt_attention_mask=warmup_prompt["attention_mask"],
        )
    print("   ✅ Compilation complete!")


# --- TTS Generation Function ---
def generate_audio_for_chunk(text_chunk, description):
    """Generate audio array for a single text chunk."""

    # Tokenise voice description (built from Cell 4 widgets)
    description_inputs = _tokenize_with_mask(description_tokenizer, description)

    # Tokenise the text to speak
    prompt_inputs = _tokenize_with_mask(prompt_tokenizer, text_chunk)

    gen_kwargs = {
        # description → text-encoder input
        "input_ids":            description_inputs["input_ids"],
        "attention_mask":       description_inputs["attention_mask"],   # ← fixed
        # text-to-speak → prompt input
        "prompt_input_ids":     prompt_inputs["input_ids"],
        "prompt_attention_mask": prompt_inputs["attention_mask"],
    }

    # Add sampling parameters
    if DO_SAMPLE:
        gen_kwargs["do_sample"]   = True
        gen_kwargs["temperature"] = TEMPERATURE
    else:
        gen_kwargs["do_sample"] = False

    with torch.no_grad():
        generation = model.generate(**gen_kwargs)

    audio_arr = generation.cpu().numpy().squeeze()
    return audio_arr


# --- Silence between chunks ---
silence = np.zeros(int(SAMPLE_RATE * SILENCE_DURATION), dtype=np.float32)

# --- Main Generation Loop ---
print(f"\n🔊 Starting TTS generation for {len(chunks)} chunks...")
print(f"   This may take a while on T4 GPU. Grab a chai ☕\n")

all_audio_segments = []
failed_chunks = []
start_time = time.time()

for i, chunk in enumerate(tqdm(chunks, desc="Generating audio")):
    try:
        audio = generate_audio_for_chunk(chunk, VOICE_DESCRIPTION)
        all_audio_segments.append(audio)
        all_audio_segments.append(silence)
    except Exception as e:
        print(f"\n⚠️  Chunk {i+1} failed: {str(e)[:100]}")
        print(f"   Skipped text: '{chunk[:60]}...'")
        failed_chunks.append(i)
        continue

elapsed = time.time() - start_time

print(f"\n✅ Generation complete in {elapsed/60:.1f} minutes!")
print(f"   Successful chunks: {len(chunks) - len(failed_chunks)}/{len(chunks)}")
if failed_chunks:
    print(f"   ⚠️  Failed chunks: {failed_chunks}")


---
## 💾 Cell 7 — Stitch, Save & Download Audio

In [ ]:
# ============================================================
# CELL 7 — Stitch audio segments, normalize, save and download
# ============================================================

from google.colab import files as colab_files
import os

if not all_audio_segments:
    raise ValueError("❌ No audio segments generated. Check Cell 6 for errors.")

# --- Concatenate all segments ---
print("🔗 Stitching audio segments...")
full_audio = np.concatenate(all_audio_segments)

# --- Normalize audio to prevent clipping ---
max_val = np.abs(full_audio).max()
if max_val > 0:
    full_audio = full_audio / max_val * 0.95

duration_sec = len(full_audio) / SAMPLE_RATE
print(f"   Duration: {duration_sec/60:.1f} min ({duration_sec:.0f} sec)")

# --- Save as WAV first ---
WAV_PATH = "/content/indic_tts_output.wav"
sf.write(WAV_PATH, full_audio, SAMPLE_RATE)

# --- Convert to MP3 if requested ---
OUTPUT_FORMAT = w_output_format.value
if OUTPUT_FORMAT == "mp3":
    from pydub import AudioSegment
    FINAL_OUTPUT_PATH = "/content/indic_tts_output.mp3"
    audio_seg = AudioSegment.from_wav(WAV_PATH)
    audio_seg.export(FINAL_OUTPUT_PATH, format="mp3", bitrate="192k")
    os.remove(WAV_PATH)
    print(f"   Converted to MP3 (192 kbps)")
else:
    FINAL_OUTPUT_PATH = WAV_PATH

file_size_mb = os.path.getsize(FINAL_OUTPUT_PATH) / (1024 * 1024)
print(f"\n✅ Saved: {FINAL_OUTPUT_PATH}")
print(f"   Size: {file_size_mb:.1f} MB")

# --- Download ---
print("\n📥 Downloading to your PC...")
colab_files.download(FINAL_OUTPUT_PATH)
print("✅ Download triggered! Check your browser's download bar.")
print(f"\n💡 Tip: Also available in Colab Files panel → {os.path.basename(FINAL_OUTPUT_PATH)}")

ValueError: ❌ No audio segments generated. Check Cell 6 for errors.

---
## 🧪 Cell 8 — Quick Test (Inline Preview)

Test a short snippet with your **current voice settings** without uploading a file.  
Edit the text and re-run to compare different voices/settings instantly.

In [ ]:
# ============================================================
# CELL 8 — Quick test with inline audio playback
# Uses ALL settings from the Voice Configuration Panel (Cell 4)
# ============================================================

from IPython.display import Audio, display

# ---- Edit your test text here ----
TEST_TEXT = "नमस्ते! यह एक test है। मैं Hindi और English दोनों बोल सकता हूँ। आज का दिन बहुत अच्छा है।"

# ---- Uses voice config from Cell 4 widgets ----
test_description = get_active_description()

print(f"🧪 Quick Test")
print(f"   Text    : '{TEST_TEXT}'")
print(f"   Speaker : {w_speaker.value}")
print(f"   Desc    : {test_description[:100]}...")
print(f"   Sample  : do_sample={w_do_sample.value}, temp={w_temperature.value}")
print("\n⏳ Generating...")

# --- Tokenise (using the same helper as Cell 6 for guaranteed attention_mask) ---
desc_inputs = _tokenize_with_mask(description_tokenizer, test_description)
text_inputs = _tokenize_with_mask(prompt_tokenizer, TEST_TEXT)

gen_kwargs = {
    "input_ids":             desc_inputs["input_ids"],
    "attention_mask":        desc_inputs["attention_mask"],   # ← fixed
    "prompt_input_ids":      text_inputs["input_ids"],
    "prompt_attention_mask": text_inputs["attention_mask"],
}
if w_do_sample.value:
    gen_kwargs["do_sample"]   = True
    gen_kwargs["temperature"] = w_temperature.value
else:
    gen_kwargs["do_sample"] = False

with torch.no_grad():
    test_generation = model.generate(**gen_kwargs)

test_audio = test_generation.cpu().numpy().squeeze()

# --- Play inline ---
print("\n▶️  Playing audio inline:")
display(Audio(test_audio, rate=SAMPLE_RATE))

# --- Save ---
test_wav = "/content/quick_test_preview.wav"
sf.write(test_wav, test_audio, SAMPLE_RATE)
duration = len(test_audio) / SAMPLE_RATE
print(f"\n💾 Saved to: {test_wav} ({duration:.1f} sec)")
print("\n💡 Tip: Change settings in Cell 4, then re-run this cell to compare voices!")


---
## 📖 Cell 9 — Reference: Speakers, Languages & Emotions

### Available Speakers by Language

| Language | Recommended | All Available |
|----------|-------------|---------------|
| **Hindi** | Rohit, Divya | Rohit, Divya, Aman, Rani |
| **English (Indian)** | Thoma, Mary | Thoma, Mary, Kabir, Priya, Raghav |
| **Bengali** | Arjun, Aditi | Arjun, Aditi, Tapan, Rashmi, Arnav, Riya |
| **Marathi** | Sanjay, Sunita | Sanjay, Sunita, Nikhil, Radha, Varun, Isha |
| **Tamil** | Kavitha | Kavitha, Jaya |
| **Telugu** | Prakash, Lalitha | Prakash, Lalitha, Kiran |
| **Gujarati** | Hardik | Hardik, Meera |
| **Kannada** | Suresh, Lakshmi | Suresh, Lakshmi, Naveen |
| **Malayalam** | Arun, Anjali | Arun, Anjali, Deepa |
| **Odia** | Bikram | Bikram, Priti |
| **Assamese** | Bhaskar | Bhaskar, Priyanka |
| **Punjabi** | Gurpreet | Gurpreet, Harleen |
| **Urdu** | Imran | Imran, Amrita |
| **Nepali** | Sagar | Sagar, Nisha |
| **Sanskrit** | Rishi | Rishi, Sita |
| **Konkani** | Karan | Karan, Leela |
| **Maithili** | Maya | Maya, Mohan |
| **Dogri** | Krishan | Krishan, Pooja |
| **Bodo** | Gyan | Gyan, Rekha |
| **Manipuri** | Tomba | Tomba, Sanahanbi |
| **Santali** | Soren | Soren, Champa |
| **Sindhi** | Govind | Govind, Sindhu |

### Emotions (Officially Supported in 10 Languages)

| Emotion | Description |
|---------|------------|
| Neutral | Default, no specific emotion |
| Narration | Storytelling, audiobook style |
| Conversation | Casual, everyday speech |
| Happy | Upbeat, cheerful |
| Sad | Subdued, melancholic |
| News | Professional, newsreader style |
| Command | Authoritative, directive |
| Anger | Intense, forceful |
| Disgust | Aversion, displeasure |
| Fear | Anxious, frightened |
| Surprise | Astonished, unexpected |

**Languages with official emotion support:** Assamese, Bengali, Bodo, Dogri, Kannada, Malayalam, Marathi, Sanskrit, Nepali, Tamil

### Voice Description Tips

- Use **"very clear audio"** for highest quality output
- Use **punctuation** (commas, periods) in your text to control prosody
- Specify **"very close"** recording for intimate, clear voice
- For **accents**: Try "A male British speaker" or "A female American speaker"
- **Emotions** work best with supported languages but can be tried with others
- Keep chunks **150-300 chars** for most natural sentence breaks

### Example Descriptions

```
Rohit's voice delivers a slightly expressive narration with a moderate pace and moderate pitch.
The recording is of very clear audio, with the speaker's voice sounding very close with no background noise.
```

```
Aditi speaks with a slightly higher pitch in a close-sounding environment. Her voice is clear,
with subtle emotional depth and a normal pace, all captured in high-quality recording.
```

```
Karan's high-pitched, engaging voice is captured in a clear, close-sounding recording.
His slightly slower delivery conveys a positive tone.
```